# Train v4+v5_1

Changes from v5_1:
- Based on val score.
- Using v4 to predict horizon=1
- Using v5_1 to predict other horizons


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))


In [ ]:
import lightgbm as lgb
import pandas as pd
import numpy as np
import gc
from src.score import weighted_rmse_score as score
import time


## Config

In [ ]:
TRAIN_PATH = PROJECT_ROOT / "data" / "train.parquet"
TEST_PATH  = PROJECT_ROOT / "data" / "test.parquet"
MODEL_DIR  = PROJECT_ROOT / "models"
SUB_DIR    = PROJECT_ROOT / "submissions"

HORIZONS = [1, 3, 10, 25]
SEEDS    = [2]
ZERO_CODES = ['MLAAMU3K', 'EP12UF2K', '1HEMHZK2', 'SJZP0OVU', '83EG83KQ']


# Per-horizon top features for industry-neutral normalization
NEUTRAL_FEATURES = {
    1:  ['feature_al', 'feature_s',  'feature_ce', 'feature_by',
         'feature_y',  'feature_t',  'feature_cf', 'feature_x', 'feature_cb'],
    3:  ['feature_ce', 'feature_cg', 'feature_v',  'feature_cf',
         'feature_y',  'feature_bg', 'feature_al', 'feature_aq', 'feature_z'],
    10: ['feature_v',  'feature_cg', 'feature_ce', 'feature_bg',
         'feature_cf', 'feature_u',  'feature_l',  'feature_z', 'feature_al'],
    25: ['feature_v',  'feature_bg', 'feature_l',  'feature_cg',
         'feature_cf', 'feature_m',  'feature_ce', 'feature_z', 'feature_aq'],
}
CAT_FEATURES = ['code', 'sub_category', 'horizon']
EXCLUDE_COLS = ['id', 'sub_code', 'ts_index', 'weight', 'y_target']
print(f'Horizons: {HORIZONS}')


Horizons: [1, 3, 10, 25]


## Feature Engineering Functions

In [15]:
def add_time_features(df):
    """Add time-index derived features."""
    ts = df['ts_index']
    df['ts_mod_7']   = ts % 5
    df['ts_mod_30']  = ts % 21
    df['ts_mod_90']  = ts % 63
    df['ts_sin']     = np.sin(2 * np.pi * ts / 252)
    df['ts_cos']     = np.cos(2 * np.pi * ts / 252)
    df['ts_sin_100'] = np.sin(2 * np.pi * ts / 100)
    df['ts_cos_100'] = np.cos(2 * np.pi * ts / 100)
    return df


def add_industry_neutral(df, horizon):
    """Add industry-neutral features: normalize by code and by sub_category, both per ts_index."""
    feats = NEUTRAL_FEATURES[horizon]
    
    for group_col, suffix in [('code', '_neut_code'), ('sub_category', '_neut_subcat')]:
        # Compute group mean and std per (group_col, ts_index)
        group_stats = df.groupby([group_col, 'ts_index'])[feats].agg(['mean', 'std'])
        # Flatten multi-level columns: (feature, mean) -> feature_mean, etc.
        group_stats.columns = [f'{f}_{s}' for f, s in group_stats.columns]
        group_stats = group_stats.reset_index()
        original_index = df.index
        # Merge  back,keep index for original df
        df = df.merge(group_stats, on=[group_col, 'ts_index'], how='left')
        df.index = original_index

        # Compute normalized features
        for f in feats:
            mean_col = f'{f}_mean'
            std_col  = f'{f}_std'
            new_col  = f'{f}{suffix}'
            df[new_col] = (df[f] - df[mean_col]) / (df[std_col] + 1e-8)
            # Drop temp columns
            df.drop(columns=[mean_col, std_col], inplace=True)
    
    return df


def create_features(df, horizon):
    """Full feature engineering pipeline for a single horizon."""
    df = add_time_features(df)
    df = add_industry_neutral(df, horizon)
    return df


def set_cat(df, cat_features):
    for feat in cat_features:
        if feat in df.columns:
            df[feat] = df[feat].astype('category')
    return df


print('Feature engineering functions defined.')


Feature engineering functions defined.


In [18]:
# Validation score
print('='*70)
print('  VALIDATION SCORES')
print('='*70)
df=pd.read_parquet(TRAIN_PATH)
SPLIT_INDEX=3500
train_mask = df['ts_index'] <= SPLIT_INDEX
val_mask = ~train_mask
df_val = df[val_mask].copy()
del df,train_mask,val_mask
gc.collect()


  VALIDATION SCORES


915

In [ ]:
val_preds_series = pd.Series(index=df_val.index, dtype=float)
horizon_scores = {}

for h in HORIZONS:
    if h ==1:
        df_val_h = df_val.query(f"horizon == {h}")
        features = [c for c in df_val_h.columns if c not in EXCLUDE_COLS]
        df_val_h = set_cat(df_val_h, CAT_FEATURES)
        model = lgb.Booster(model_file=f"{MODEL_DIR}/lgb_model_v4_horizon{h}.txt")
        val_preds = model.predict(df_val_h[features], num_iteration=model.best_iteration)
        val_score = score(df_val_h['y_target'], val_preds, df_val_h['weight'])
        horizon_scores[h] = val_score
        val_preds_series[df_val_h.index] = val_preds
        print(f"Horizon {h:>2d}: Validation WRMSE = {val_score:.6f}")
    else:
        df_val_h = create_features(
            df_val.query(f"horizon == {h}"),
            horizon=h
        )
        features = [c for c in df_val_h.columns if c not in EXCLUDE_COLS]
        df_val_h = set_cat(df_val_h, CAT_FEATURES)
        model = lgb.Booster(model_file=f"{MODEL_DIR}/lgb_model_v5_1_horizon{h}_seed{2}.txt")
        val_preds = model.predict(df_val_h[features], num_iteration=model.best_iteration)
        val_score = score(df_val_h['y_target'], val_preds, df_val_h['weight'])
        horizon_scores[h] = val_score
        val_preds_series[df_val_h.index] = val_preds
        print(f"Horizon {h:>2d}: Validation WRMSE = {val_score:.6f}")
print('\nOverall Validation WRMSE:', score(df_val['y_target'], val_preds_series, df_val['weight']))


Horizon  1: Validation WRMSE = 0.089991
Horizon  3: Validation WRMSE = 0.140521
Horizon 10: Validation WRMSE = 0.237511
Horizon 25: Validation WRMSE = 0.310505

Overall Validation WRMSE: 0.26253353585176853


## Test Predictions
- For Horizon=1 : using v4

- For Horizon=3,10,25:using v5_1

In [ ]:
#- For Horizon=1 : using v4
#For Horizon=3,10,25:using v5_1
test_preds_parts = []  # list of DataFrames with (id, prediction)

for h in HORIZONS:
    t0 = time.time()
    print(f"\n  Predicting horizon {h}...")
    if h==1:
        model=lgb.Booster(model_file=f"{MODEL_DIR}/lgb_model_v4_horizon{h}.txt")
        df_test_h = pd.read_parquet(TEST_PATH).query(f"horizon == {h}").reset_index(drop=True)
        features = [c for c in df_test_h.columns if c not in EXCLUDE_COLS]
        df_test_h = set_cat(df_test_h, CAT_FEATURES)
        preds = model.predict(df_test_h[features], num_iteration=model.best_iteration)
        zero_mask = df_test_h['code'].isin(ZERO_CODES)
        preds[zero_mask] = 0.0
    else:
        df_test_h = create_features(
            pd.read_parquet(TEST_PATH).query(f"horizon == {h}").reset_index(drop=True),
            horizon=h
        )
        features = [c for c in df_test_h.columns if c not in EXCLUDE_COLS]
        df_test_h = set_cat(df_test_h, CAT_FEATURES)
        model = lgb.Booster(model_file=f"{MODEL_DIR}/lgb_model_v5_1_horizon{h}_seed{2}.txt")
        preds = model.predict(df_test_h[features], num_iteration=model.best_iteration)

        # Zero-code override
        zero_mask = df_test_h['code'].isin(ZERO_CODES)
        preds[zero_mask] = 0.0

    test_preds_parts.append(pd.DataFrame({
        'id': df_test_h['id'].values,
        'prediction': preds
    }))

    print(f"    Horizon {h:>2d}: {len(df_test_h):,} rows, "
          f"zero-coded: {zero_mask.sum():,}  ({time.time()-t0:.1f}s)")

    del df_test_h
    gc.collect()

# Combine
prediction_df = pd.concat(test_preds_parts, ignore_index=True)
assert prediction_df['prediction'].isna().sum() == 0, 'NaN in predictions!'
print(f"\n  Total test predictions: {len(prediction_df):,}")
prediction_df.head()



  Predicting horizon 1...
    Horizon  1: 379,617 rows, zero-coded: 69,005  (34.1s)

  Predicting horizon 3...
    Horizon  3: 376,558 rows, zero-coded: 68,523  (61.3s)

  Predicting horizon 10...
    Horizon 10: 362,057 rows, zero-coded: 65,675  (27.3s)

  Predicting horizon 25...
    Horizon 25: 328,875 rows, zero-coded: 59,033  (34.3s)

  Total test predictions: 1,447,107


,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.006606
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3648,-0.004620
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3649,-0.007292
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3650,-0.008612
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3651,-0.011215


## Save Submission

In [21]:
sub_path = f"{SUB_DIR}/submission_combined_v4_v5_1.csv"
prediction_df.to_csv(sub_path, index=False)
print(f"Submission saved -> {sub_path}")
prediction_df.info()
prediction_df.head()


Submission saved -> /home/lingod/kaggle_projects/ts_forecasting/submissions/submission_combined_v4_v5_1.csv
<class 'pandas.DataFrame'>
RangeIndex: 1447107 entries, 0 to 1447106
Data columns (total 2 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   id          1447107 non-null  str    
 1   prediction  1447107 non-null  float64
dtypes: float64(1), str(1)
memory usage: 74.0 MB


,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.006606
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3648,-0.004620
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3649,-0.007292
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3650,-0.008612
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3651,-0.011215


## (Optional) Reload Models from Disk

In [ ]:
# all_models = {}
# for h in HORIZONS:
#     for seed in SEEDS:
#         path = f"{MODEL_DIR}/lgb_v5_h{h}_seed{seed}.txt"
#         all_models[(h, seed)] = lgb.Booster(model_file=path)
# print(f'Reloaded {len(all_models)} models.')
